# 3. Explain an ONNX probe with DiDAE, without training anything

This is the README walkthrough *"explain an ONNX classifier or foundation-model probe end to end"*
as a notebook.

The difference to notebook 2 is that **nothing is trained here**. DiDAE reuses a pretrained generator
and a pretrained sparse dictionary, distils your model into the generator's encoder space in closed
form, edits along dictionary directions, decodes, and re-classifies with your model. Your model is
only ever called forward, which is why an ONNX file is enough.

The generator's encoder and your model's encoder need not be the same network: the paper's ImageNet
experiment ranks a DINOv3 probe against a dictionary that lives in CLIP space.

In [ ]:
import os, subprocess, sys, textwrap
from pathlib import Path

# PEAL resolves <PEAL_BASE> itself; $PEAL_DATA and $PEAL_RUNS default to ./datasets and ./peal_runs.
PEAL_BASE = Path(os.environ.get("PEAL_BASE", Path.cwd().parent)).resolve()
PEAL_DATA = Path(os.environ.get("PEAL_DATA", PEAL_BASE / "datasets"))
PEAL_RUNS = Path(os.environ.get("PEAL_RUNS", PEAL_BASE / "peal_runs"))
os.chdir(PEAL_BASE)          # every entry point expects the repo root as the working directory
# Export them: os.path.expandvars and the PEAL entry points both read the environment,
# so a value only held in a Python variable leaves "$PEAL_RUNS" unexpanded in paths.
os.environ["PEAL_DATA"] = str(PEAL_DATA)
os.environ["PEAL_RUNS"] = str(PEAL_RUNS)
print("PEAL_BASE", PEAL_BASE, "\nPEAL_DATA", PEAL_DATA, "\nPEAL_RUNS", PEAL_RUNS)

def run(cmd):
    """Stream a PEAL command. These are long jobs; the output is the run log."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


In [ ]:
import yaml

def patch_yaml(template, target, **changes):
    """Copy a shipped config and overwrite single top-level keys.

    Comments are lost (yaml round-trips values only), which is why the README tells you to copy the
    files by hand. Doing it programmatically keeps a notebook reproducible; open the template if you
    want to read what each field means.
    """
    src = PEAL_BASE / template
    dst = PEAL_BASE / target
    dst.parent.mkdir(parents=True, exist_ok=True)
    cfg = yaml.safe_load(src.read_text())
    for k, v in changes.items():
        cfg[k] = v
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"wrote {target}")
    return dst


## Step 1 — export and sanity-check the probe

Same contract as notebook 2: the graph is handed the dataset tensor unchanged, and the batch axis
must be dynamic.

In [ ]:
DATASET = "my_data"
onnx_path = PEAL_RUNS / "my_model" / "model.onnx"
# Export your model first, exactly as in notebook 2. Nothing below works without it.
if onnx_path.exists():
    # onnx / onnxruntime are NOT in the base `peal` environment. Install the web extras
    # (pip install -r requirements-web.txt) or use the .venv-web interpreter.
    import numpy as np, onnxruntime as ort

    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    out = sess.run(None, {name: np.random.rand(2, 3, 224, 224).astype(np.float32)})[0]
    print("output", out.shape, "- logits per image:", out.shape[-1])
else:
    raise FileNotFoundError(
        f"export your model to {onnx_path} first (see notebook 2); the rest of this "
        "notebook has nothing to explain without it"
    )


## Step 1b — the dataset and its config

This notebook trains nothing, but DiDAE still has to read your images: the sweep needs samples to
edit, and the config needs a `data:` entry. `Image2MixedDataset` wants `$PEAL_DATA/<name>/imgs/...`
plus a `data.csv` whose first column is each image's path relative to `imgs`, followed by one column
per label. The cell below writes that csv and derives the data config; adapt `label_of()` to your own
labelling rule.


In [ ]:
import csv

SOURCE_DIR = PEAL_DATA / DATASET / "imgs"

def label_of(path):
    """Return 0 or 1 for an image path. Replace with your own rule."""
    return 1 if path.parent.name.lower().startswith("positive") else 0

rows = [(str(pp.relative_to(SOURCE_DIR)), label_of(pp))
        for pp in sorted(SOURCE_DIR.rglob("*"))
        if pp.suffix.lower() in {".png", ".jpg", ".jpeg"}]
if not rows:
    raise FileNotFoundError(f"no images under {SOURCE_DIR}")
csv_path = PEAL_DATA / DATASET / "data.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["imgs", "Label1"]); w.writerows(rows)
print(f"{len(rows)} images -> {csv_path}")
print("class balance:", {c: sum(1 for r in rows if r[1] == c) for c in (0, 1)})

# The Image2MixedDataset template that actually reads a data.csv. Do not use
# configs/sce_experiments/data/imagenet.yaml: that one is an ImageFolder loader with 1000 outputs.
data_cfg = patch_yaml(
    "configs/sce_experiments/data/imagenet_husky_vs_wulf_resnet18.yaml",
    f"configs/my_experiments/data/{DATASET}.yaml",
    dataset_path=f"$PEAL_DATA/{DATASET}",
)
cfg = yaml.safe_load(data_cfg.read_text())
cfg.pop("dataset_class", None); cfg.pop("confounding_factors", None)
# input_size and normalization must match what your ONNX model expects.
cfg["input_size"] = [3, 224, 224]
data_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("data config:", data_cfg)


## Step 2 — choose the reference generator and dictionary

These are the two pretrained components DiDAE composes. For natural images: the representation
autoencoder over OpenAI CLIP ViT-L/14, plus the public 6144-atom MSAE. Neither is trained here; the
generator path points at a decoding preset written next to the checkpoint (they differ only in
epoch, sampler and guidance scale).

In [ ]:
# The preset under $PEAL_RUNS/imagenet/rae_clip/ is NOT portable: it pins an absolute
# checkpoint path from the machine it was trained on. This config resolves its weights
# from $PEAL_RAE_WEIGHTS instead (an hf://<repo> or a local folder with decoder.pt,
# stats.pt and stage2_ema.pt; see tools/export_rae_weights.py).
# It is a RAEDiffusionAutoencoder, so it also needs the separate RAEv2 install:
#     python tools/install_rae.py
GENERATOR  = "<PEAL_BASE>/configs/web_demo/imagenet_rae_clip_hf.yaml"
DICTIONARY = "<PEAL_BASE>/configs/didae_experiments/sparse_dictionaries/msae_decomposition.yaml"

for label, ref in (("generator", GENERATOR), ("dictionary", DICTIONARY)):
    path = Path(os.path.expandvars(ref.replace("<PEAL_BASE>", str(PEAL_BASE))))
    print(f"{label} config present:", path.exists(), "\n ", path)
print("RAE weights ($PEAL_RAE_WEIGHTS):", os.environ.get("PEAL_RAE_WEIGHTS", "NOT SET"))


## Step 3 — the adaptor config

Copy a shipped DiDAE config and change four paths. Useful knobs: `top_k_directions` caps how many
directions get decoded and shown, `n_samples` how many images the sweep runs over, and
`sweep_atom_subset` restricts the sweep to named atoms when you already suspect one.

`finetune_iterations: 0` stops after the explanation. Leave it there for now — see the caveat at the
end about repairing an ONNX model.

In [ ]:
patch_yaml(
    "configs/didae_experiments/adaptors/imagenet_koala_vs_wombat_dinov3_ep43s100_msae.yaml",
    f"configs/my_experiments/adaptors/{DATASET}_didae.yaml",
    student=str(onnx_path).replace(str(PEAL_RUNS), "$PEAL_RUNS"),
    data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}.yaml",
    base_dir=f"$PEAL_RUNS/my_model/didae",
    # The template carries a teacher checkpoint and a test_data config that exist only
    # on our machine. DiDAE.__init__ builds the teacher and resolves test_data BEFORE it
    # looks at finetune_iterations, so leaving them alone fails on construction even with
    # finetune_iterations=0.
    test_data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}.yaml",
    teacher="human@8000",
    generator=GENERATOR,
    sparse_dictionary=DICTIONARY,
    top_k_directions=10,
    finetune_iterations=0,
    task={"criterions": {"ce": 1.0}, "output_type": "singleclass",
          "output_channels": 2, "y_selection": ["Label1"]},
)


## Step 4 — run the sweep

In [ ]:
run(f'python -W ignore run_didae.py --config "<PEAL_BASE>/configs/my_experiments/adaptors/{DATASET}_didae.yaml"')

## Step 5 — read the ranking

`sweep_results.pt` holds every candidate direction with its latent, ambient and verified flip counts.
The ranking key is **verified** flips: a direction counts only when the decoded image flips your
model *and* a re-encoding confirms the intended concept moved.

In [ ]:
import torch
RUN = PEAL_RUNS / "my_model" / "didae"
sw = RUN / "sweep_results.pt"
if sw.exists():
    res = torch.load(sw, map_location="cpu", weights_only=False)
    print(type(res), len(res) if hasattr(res, "__len__") else "")
    # entries key the atom as `dimension_name` and `direction_idx`, not `component_idx`
    rows = res if isinstance(res, list) else res.get("directions", res)
    for r in list(rows)[:10]:
        if isinstance(r, dict):
            print({k: r[k] for k in list(r)[:6] if k in r})
else:
    print("no sweep_results.pt yet at", sw)


### Check the distilled probe before trusting the ranking

DiDAE ranks directions by their effect on a linear probe fitted to your model in the encoder space.
If that probe does not agree with your model, the ranking describes the probe, not the model.

In [ ]:
dp = RUN / "distilled_probe"
print("distilled probe dir:", dp, "exists:", dp.exists())
if dp.exists():
    print(sorted(p.name for p in dp.iterdir())[:10])
    print("\nCompare its agreement with your ONNX model on held-out images before reading the ranking.")


## Step 6 — the evidence, as before/after pairs

Each ranked direction gets a folder of factual/counterfactual collages, best first. This is what a
teacher looks at to decide whether a direction is a genuine class feature or a shortcut.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

coll = sorted((RUN / "direction_collages").glob("rank*")) if (RUN / "direction_collages").exists() else []
print(f"{len(coll)} ranked directions with collages")
for d in coll[:3]:
    imgs = sorted(d.glob("*.png"))[:1]
    for p in imgs:
        plt.figure(figsize=(7, 3)); plt.imshow(Image.open(p)); plt.axis("off")
        plt.title(d.name, fontsize=9); plt.show()


## Repairing the model

The teacher decides which directions are spurious. `teacher: "cluster@8000"` serves the collages at
`http://localhost:8000` for hand labelling; a path to an unpoisoned `.cpl` uses that model as an
oracle. Counterfactuals along directions marked `false` become a corrective training set.

**The limitation to plan around:** an ONNX student can usually be fine-tuned as well. PEAL converts
the graph into a trainable torch module with `onnx2torch`, which is what `ModelTrainer` needs. The
conversion is not universal: a graph using an operator `onnx2torch` does not implement falls back to
an inference-only `onnxruntime` closure, with a warning naming the operator, and then it can be
explained and ranked but **not** fine-tuned. The DINOv3 probes from the ImageNet experiments are such
a case, on a `Size` node. Check the warning for your own model. If it does fall back, either stop here with
`finetune_iterations: 0`, or point `student` at the same model as a PyTorch `.cpl` for the repair and
keep the ONNX file for the sweep. With a `.cpl` student,
`continuous_learning: "deep_feature_reweighting"` retrains only the last layer, which is usually what
you want for a probe.

In [ ]:
REPAIR = False   # set True once `student` points at a PyTorch .cpl
if REPAIR:
    patch_yaml(f"configs/my_experiments/adaptors/{DATASET}_didae.yaml",
               f"configs/my_experiments/adaptors/{DATASET}_didae_repair.yaml",
               student=f"$PEAL_RUNS/my_model/model.cpl",
               teacher="cluster@8000",
               finetune_iterations=1,
               continuous_learning="deep_feature_reweighting",
               base_dir=f"$PEAL_RUNS/my_model/didae_repair")
    run(f'python -W ignore run_didae.py --config "<PEAL_BASE>/configs/my_experiments/adaptors/{DATASET}_didae_repair.yaml"')
